# Parse Optimized Contraction Files

Parse `*_mom_optimized.txt` files and extract Phase 1 (matrix-matrix) product definitions into structured Python dicts.

In [1]:
import re
from pprint import pprint

In [2]:
def parse_pi_operand(text):
    """Parse a single operand like 'Pi(-k, t_snk + Delta)' or 'prod_Pi3'.

    Returns a dict. For a Pi field:
        {'type': 'source', 'momentum': '-k', 'time': 't_snk + Delta'}
    For a reference to another product:
        {'type': 'product', 'ref': 'prod_Pi3'}
    """
    # will search for source fields
    pi_match = re.match(r'Pi\((.+),\s*(.+)\)', text.strip())
    if pi_match:
        return {
            # source = MesonField or Prod (product of meson fields from earlier level)
            'type': 'source',
            'momentum': pi_match.group(1).strip(),
            'time': pi_match.group(2).strip()
        }
        
    # if not a source field then it searches for products
    prod_match = re.match(r'(prod_Pi\d+)', text.strip())
    if prod_match:
        return {'type': 'product', 'ref': prod_match.group(1)}

    raise ValueError(f"Could not parse operand: '{text}'")


def parse_product_line(line):
    """Parse a line like 'prod_Pi1 = Pi(-k, t_snk + Delta) . Pi(-p, t_src + Delta)'.

    Returns (name, left_operand_dict, right_operand_dict).
    """
    # .split(#) splits the string at '#' character
    # [0] takes the first part of the string, ie bit before #
    # .strip() removed whitespace
    line = line.split('#')[0].strip()

    # .split('=', 1) means to only split at the first occurance of '=' in the string
    name, rhs = line.split('=', 1)
    # ie Prod_Pi1 and .strip() removes whitespace
    name = name.strip()
    # just gets the rhs
    rhs = rhs.strip()
    # split into two meson fields on eiter end of the \cdot
    parts = rhs.split(' . ')
    if len(parts) != 2:
        raise ValueError(f"Expected exactly 2 operands in: '{line}'")

    #  Ex argument: Pi(p, tsrc)
    # the individual Pis are to be passed to the above function
    left = parse_pi_operand(parts[0])
    
    right = parse_pi_operand(parts[1])

    return name, left, right


def parse_phase1(filepath):
    """Parse Phase 1 (matrix-matrix products) from an optimized contraction file.

    Returns:
        level1: dict of dicts for Level 1 products
        level2: dict of dicts for Level 2 products (may be empty)
    """
    level1 = {}
    level2 = {}

    current_level = None

    with open(filepath) as f:
        for line in f:
            stripped = line.strip()

            if stripped == '# Level 1:':
                current_level = 1
                continue
            if stripped == '# Level 2:':
                current_level = 2
                continue

            if 'Phase 2' in stripped or 'Phase 3' in stripped:
                current_level = None
                continue

            if stripped.startswith('#') or stripped == '':
                continue

            if current_level and stripped.startswith('prod_Pi'):
                name, left, right = parse_product_line(stripped)
                product = {'left': left, 'right': right}

                if current_level == 1:
                    level1[name] = product
                elif current_level == 2:
                    level2[name] = product

    return level1, level2

## Zeroth Order

In [3]:
level1, level2 = parse_phase1('I2_pipi_cexpr_mom_optimized.txt')

print(f'Level 1: {len(level1)} products')
pprint(level1)

if level2:
    print(f'\nLevel 2: {len(level2)} products')
    pprint(level2)
else:
    print('\nNo Level 2 products')

Level 1: 4 products
{'prod_Pi1': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': '-p',
                        'time': 't_src + Delta',
                        'type': 'source'}},
 'prod_Pi2': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': 'p', 'time': 't_src', 'type': 'source'}},
 'prod_Pi3': {'left': {'momentum': 'k', 'time': 't_snk', 'type': 'source'},
              'right': {'momentum': '-p',
                        'time': 't_src + Delta',
                        'type': 'source'}},
 'prod_Pi4': {'left': {'momentum': 'k', 'time': 't_snk', 'type': 'source'},
              'right': {'momentum': 'p', 'time': 't_src', 'type': 'source'}}}

No Level 2 products


## EM Corrections

In [4]:
level1_em, level2_em = parse_phase1('I2_pipi_EM_cexpr_mom_optimized.txt')

print(f'Level 1: {len(level1_em)} products')
pprint(level1_em)

print(f'\nLevel 2: {len(level2_em)} products')
pprint(level2_em)

Level 1: 8 products
{'prod_Pi1': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': '-p',
                        'time': 't_src + Delta',
                        'type': 'source'}},
 'prod_Pi2': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': 'p', 'time': 't_src', 'type': 'source'}},
 'prod_Pi3': {'left': {'momentum': '-p',
                       'time': 't_src + Delta',
                       'type': 'source'},
              'right': {'momentum': '-k',
                        'time': 't_snk + Delta',
                        'type': 'source'}},
 'prod_Pi4': {'left': {'momentum': '-p',
                       'time': 't_src + Delta',
                       'type': 'source'},
              'right': {'momentum': 'k', 'time': 't_snk', 'type': 'source'}},
 'prod_Pi5': {'left': {'moment

## Explore the data structure

Play around with accessing individual products and their operands.

In [5]:
# Access a single Level 1 product
print('prod_Pi1:', level1_em['prod_Pi1'])
print()
print('left operand:', level1_em['prod_Pi1']['left'])
print('left momentum:', level1_em['prod_Pi1']['left']['momentum'])
print('left time:', level1_em['prod_Pi1']['left']['time'])

prod_Pi1: {'left': {'type': 'source', 'momentum': '-k', 'time': 't_snk + Delta'}, 'right': {'type': 'source', 'momentum': '-p', 'time': 't_src + Delta'}}

left operand: {'type': 'source', 'momentum': '-k', 'time': 't_snk + Delta'}
left momentum: -k
left time: t_snk + Delta


In [6]:
# Access a Level 2 product — notice operands can reference Level 1 results
print('prod_Pi10:', level2_em['prod_Pi10'])
print()
print('left type:', level2_em['prod_Pi10']['left']['type'])   # 'product'
print('left ref:', level2_em['prod_Pi10']['left']['ref'])      # 'prod_Pi1'
print('right type:', level2_em['prod_Pi10']['right']['type'])  # 'product'
print('right ref:', level2_em['prod_Pi10']['right']['ref'])    # 'prod_Pi6'

prod_Pi10: {'left': {'type': 'product', 'ref': 'prod_Pi1'}, 'right': {'type': 'product', 'ref': 'prod_Pi6'}}

left type: product
left ref: prod_Pi1
right type: product
right ref: prod_Pi6


In [7]:
# Loop over all Level 1 products
for name, product in level1_em.items():
    L = product['left']
    R = product['right']
    print(f"{name}: Pi({L['momentum']}, {L['time']}) . Pi({R['momentum']}, {R['time']})")

prod_Pi1: Pi(-k, t_snk + Delta) . Pi(-p, t_src + Delta)
prod_Pi2: Pi(-k, t_snk + Delta) . Pi(p, t_src)
prod_Pi3: Pi(-p, t_src + Delta) . Pi(-k, t_snk + Delta)
prod_Pi4: Pi(-p, t_src + Delta) . Pi(k, t_snk)
prod_Pi5: Pi(k, t_snk) . Pi(-p, t_src + Delta)
prod_Pi6: Pi(k, t_snk) . Pi(p, t_src)
prod_Pi7: Pi(p, t_src) . Pi(-k, t_snk + Delta)
prod_Pi8: Pi(p, t_src) . Pi(k, t_snk)


In [8]:
# Check which Level 2 products have both operands as Level 1 results
# (these are the ones that need the fix in MesonField_MesonField_connected)
for name, product in level2_em.items():
    L = product['left']
    R = product['right']
    if L['type'] == 'product' and R['type'] == 'product':
        print(f"{name}: {L['ref']} . {R['ref']}  <-- both from Level 1 results")

prod_Pi10: prod_Pi1 . prod_Pi6  <-- both from Level 1 results
prod_Pi12: prod_Pi2 . prod_Pi5  <-- both from Level 1 results
prod_Pi14: prod_Pi3 . prod_Pi8  <-- both from Level 1 results
prod_Pi16: prod_Pi4 . prod_Pi7  <-- both from Level 1 results
prod_Pi18: prod_Pi5 . prod_Pi2  <-- both from Level 1 results
prod_Pi20: prod_Pi6 . prod_Pi1  <-- both from Level 1 results
prod_Pi22: prod_Pi7 . prod_Pi4  <-- both from Level 1 results
prod_Pi24: prod_Pi8 . prod_Pi3  <-- both from Level 1 results


In [9]:
pprint(level1)

{'prod_Pi1': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': '-p',
                        'time': 't_src + Delta',
                        'type': 'source'}},
 'prod_Pi2': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': 'p', 'time': 't_src', 'type': 'source'}},
 'prod_Pi3': {'left': {'momentum': 'k', 'time': 't_snk', 'type': 'source'},
              'right': {'momentum': '-p',
                        'time': 't_src + Delta',
                        'type': 'source'}},
 'prod_Pi4': {'left': {'momentum': 'k', 'time': 't_snk', 'type': 'source'},
              'right': {'momentum': 'p', 'time': 't_src', 'type': 'source'}}}


In [10]:
level1['prod_Pi1']['left']['momentum']

'-k'

In [11]:
mom_map = {
    'p': 1,
    # recall that these are indices in a Eigen tensor of Mesonfield(p, t, i, j)
    '-p': 2,
    'k': 3,
    '-k': 4
}

In [12]:
level1, level2 = parse_phase1('I2_pipi_cexpr_mom_optimized.txt')

print(f'Level 1: {len(level1)} products')
pprint(level1)

if level2:
    print(f'\nLevel 2: {len(level2)} products')
    pprint(level2)

Level 1: 4 products
{'prod_Pi1': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': '-p',
                        'time': 't_src + Delta',
                        'type': 'source'}},
 'prod_Pi2': {'left': {'momentum': '-k',
                       'time': 't_snk + Delta',
                       'type': 'source'},
              'right': {'momentum': 'p', 'time': 't_src', 'type': 'source'}},
 'prod_Pi3': {'left': {'momentum': 'k', 'time': 't_snk', 'type': 'source'},
              'right': {'momentum': '-p',
                        'time': 't_src + Delta',
                        'type': 'source'}},
 'prod_Pi4': {'left': {'momentum': 'k', 'time': 't_snk', 'type': 'source'},
              'right': {'momentum': 'p', 'time': 't_src', 'type': 'source'}}}


## Messing around with translating the above dicts to Mesonfield 
- Function that maps (p,t) index to flattened MesonField index
- Going to want to make these map and time dictionaries to translate the outputs from the above into momenta and time indices
- Also 

In [94]:
def momentum_time_index_to_flattened_index(p, t, Nmodes, Nt):
    return p * Nt * Nmodes * Nmodes + t * Nmodes * Nmodes

In [97]:

def make_momentum_map(p, min_p, k, min_k):
    return { # recall that these are indices in a Eigen tensor of Mesonfield(p, t, i, j)
            'p': p,
            '-p': min_p,
            'k': k,
            '-k': min_k
            }

def make_time_map(tsrc, tsnk, Delta):
    return {
             't_src': tsrc,
             't_snk': tsnk,
             't_src + Delta': tsrc + Delta,
             't_snk + Delta': tsnk + Delta
           }

In [98]:
#make_momentum_map(1,1)[level1['prod_Pi1']['left']['momentum']]

level1['prod_Pi1']#['left']['momentum']

for i, kk in enumerate(level1):
    pprint(i)
    pprint(kk)
#    for j in level1[i]:
        #make_momentum_map(1,1)[ pprint(level1[i][j]['momentum']) ]
        #pprint(level1[i][j]['momentum'])
#        pprint(j)

0
'prod_Pi1'
1
'prod_Pi2'
2
'prod_Pi3'
3
'prod_Pi4'


In [110]:
                            
def level1_to_contractions(momenta_set,  # numerical values for [p,k]
                           time_set,  # numerical values [tsrc, tsnk, Delta] = [tsrc ,tsrc + dt, Delta]
                           level1,  # parsed level 1 above
                           Nmodes, 
                           Nt,
                           Ac = [],
                           Bc = [],
                           Cc = []): # contraction lists
    Aadd = []
    Badd = []
    Cadd = {}
    
    momenta_map = make_momentum_map(momenta_set[0], momenta_set[1], momenta_set[2], momenta_set[3])
    time_map = make_time_map(time_set[0], time_set[1], time_set[2])

    for n, i in enumerate(level1): # first level products
        momentum = momenta_map[level1[i]['left']['momentum']]
        time = time_map[level1[i]['left']['time']]

        Aadd.append( momentum_time_index_to_flattened_index(momentum, time, Nmodes, Nt) )

        momentum = momenta_map[level1[i]['right']['momentum']]
        time = time_map[level1[i]['right']['time']]

        Badd.append( momentum_time_index_to_flattened_index(momentum, time, Nmodes, Nt) )

        Cadd[i] = n # C['Prod_Pi1'] = 0, C['Prod_Pi2'] = 1 , etc
    
    if(len(Ac) == 0):
        return [Aadd, Badd, Cadd]

    Cc |= Cadd
    
    return [Ac + Aadd, Bc + Badd, Cc]

In [111]:
# will need to do all the products for a specified k,p,t_src,t_snk before going onto next one

In [112]:
mom = [1,1, 2,2]
time = [0, 2, 2]
level1_to_contractions(mom, time, level1, 2000, 32)

[[272000000, 272000000, 264000000, 264000000],
 [136000000, 128000000, 136000000, 128000000],
 {'prod_Pi1': 0, 'prod_Pi2': 1, 'prod_Pi3': 2, 'prod_Pi4': 3}]

In [39]:
# will want to add matrices to the device with a while (matrices < max_matrices_per_device)
Nmodes = 2000

GPU_mem = 40
safe_GPU_mem = GPU_mem * 0.8

individual_matrix_size_gib = Nmodes * Nmodes * 16 / 1024**3

max_matrices_per_device = safe_GPU_mem / individual_matrix_size_gib // 3 # divided by three because I need mem for A,B and C

p_max = 4
k_max = 4
tsrc_max = 32
tsrc_sample_freq = 6 
delta_t_max = 14

p_list = [i for i in range(p_max)]
k_list = [i for i in range(k_max)]
t_src_list = [int(i*tsrc_sample_freq) for i in range(tsrc_max // (tsrc_sample_freq - 1)) ]

delta_t_list = [i for i in range(delta_t_max)]

In [40]:
t_src_list

[0, 6, 12, 18, 24, 30]

In [ ]:
# try to translate all the products in level one into A_contractions and B_contractions
